# Stage 4 Recovery — Resume from saved bertopic_fine

The previous run saved `bertopic_fine`, `topic_keywords_fine.csv`, and `hierarchical_topics.parquet`
before the kernel was OOM-killed during outlier rescue.

This notebook skips fit_transform entirely — topic assignments are loaded directly
from the saved model. Outlier rescue uses embedding cosine similarity instead of
approximate_distribution (which OOM-killed the previous run on 467k outliers).

**Datasets needed:**
- `ns-sentiment-chunks` — submissions_chunks.parquet + comments_chunks.parquet
- `ns-sentiment-bertopic-fine` — the bertopic_fine folder from the previous run

**Run via Save & Run All (committed mode).**

In [ ]:
# Cell 1 — Discover exact input paths
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# Cell 2 — Config  ← UPDATE PATHS AFTER RUNNING CELL 1
SUBMISSIONS_CHUNKS = '/kaggle/input/<your-dataset>/submissions_chunks.parquet'
COMMENTS_CHUNKS    = '/kaggle/input/<your-dataset>/comments_chunks.parquet'
BERTOPIC_FINE_PATH = '/kaggle/input/<your-bertopic-dataset>/bertopic_fine'
OUT_DIR            = '/kaggle/working'

NR_COARSE_TOPICS   = 20
OUTLIER_MIN_SIM    = 0.3   # cosine similarity threshold for outlier rescue
RESCUE_BATCH_SIZE  = 10_000  # process outliers in batches to avoid OOM

_SINGLISH_STOP = ['lah', 'lor', 'leh', 'sia', 'meh', 'hor', 'wah', 'ah']

In [ ]:
# Cell 3 — Install + imports
!pip install bertopic safetensors sentence-transformers -q

import json
import numpy as np
import pandas as pd
from bertopic import BERTopic
from sklearn.preprocessing import normalize

print('Imports OK')

In [ ]:
# Cell 4 — Load chunks and stack embeddings
sub_cols = ['chunk_id', 'doc_id', 'text', 'embedding', 'subreddit',
            'created_utc', 'score', 'log_weight', 'doc_type']
com_cols = sub_cols + ['depth', 'post_id']

sub = pd.read_parquet(SUBMISSIONS_CHUNKS, columns=sub_cols)
com = pd.read_parquet(COMMENTS_CHUNKS, columns=com_cols)
com['created_utc'] = com['created_utc'].astype('datetime64[ms, UTC]')

combined = pd.concat([sub, com], ignore_index=True)
print(f'Total chunks: {len(combined):,}')

first = combined['embedding'].iloc[0]
embeddings = (
    np.stack(combined['embedding'].values).astype(np.float32)
    if isinstance(first, np.ndarray)
    else np.array(combined['embedding'].tolist(), dtype=np.float32)
)
print(f'Embedding matrix: {embeddings.shape}')
docs = combined['text'].tolist()

In [ ]:
# Cell 5 — Load saved fine model and recover topic assignments
#
# BERTopic's safetensors format saves per-document topic assignments in
# topics.json under the 'topics' key — no need to rerun fit_transform.

topic_model = BERTopic.load(BERTOPIC_FINE_PATH)

topics_raw = np.array(topic_model.topics_, dtype=np.int16)
assert len(topics_raw) == len(combined), \
    f'Mismatch: model has {len(topics_raw)} assignments, combined has {len(combined)} chunks'

n_fine    = len(set(topics_raw)) - (1 if -1 in topics_raw else 0)
n_outlier = (topics_raw == -1).sum()
print(f'Fine topics: {n_fine}')
print(f'Outliers:    {n_outlier:,} ({n_outlier / len(topics_raw) * 100:.1f}%)')

In [ ]:
# Cell 6 — Outlier rescue via embedding cosine similarity
#
# For each outlier chunk, find the topic whose centroid embedding is most
# similar (cosine) to the chunk's embedding. Assign if similarity >= threshold.
#
# Processes outliers in batches of RESCUE_BATCH_SIZE to avoid OOM.
# This replaces approximate_distribution which OOM-killed the previous run.

topics_fine = topics_raw.copy()
probs_fine  = np.where(topics_raw != -1, 1.0, 0.0).astype(np.float32)

outlier_idx = np.where(topics_fine == -1)[0]
print(f'Rescuing {len(outlier_idx):,} outliers in batches of {RESCUE_BATCH_SIZE:,} …')

# topic_embeddings_ shape: (n_topics, 768) — centroids in original embedding space
topic_embs   = np.array(topic_model.topic_embeddings_, dtype=np.float32)
unique_topics = sorted(t for t in set(topics_raw) if t != -1)

# Normalise topic centroids once
topic_embs_norm = normalize(topic_embs, norm='l2')

rescued = 0
for batch_start in range(0, len(outlier_idx), RESCUE_BATCH_SIZE):
    batch_idx = outlier_idx[batch_start : batch_start + RESCUE_BATCH_SIZE]
    batch_embs = embeddings[batch_idx]                    # (batch, 768)
    batch_norm = normalize(batch_embs, norm='l2')         # normalise for cosine
    sims       = batch_norm @ topic_embs_norm.T           # (batch, n_topics)

    best_col  = sims.argmax(axis=1)
    best_sim  = sims.max(axis=1)

    for i, (col, sim) in enumerate(zip(best_col, best_sim)):
        if sim >= OUTLIER_MIN_SIM:
            topics_fine[batch_idx[i]] = unique_topics[col]
            probs_fine[batch_idx[i]]  = float(sim)
            rescued += 1

    if (batch_start // RESCUE_BATCH_SIZE + 1) % 10 == 0:
        print(f'  Processed {batch_start + len(batch_idx):,} / {len(outlier_idx):,}')

remaining = (topics_fine == -1).sum()
print(f'Rescued {rescued:,} / {len(outlier_idx):,}  |  {remaining:,} remain as -1')

In [ ]:
# Cell 7 — Coarse reduction to NR_COARSE_TOPICS
#
# reduce_topics() cuts the hierarchical dendrogram and modifies the model in-place.
# Rescued outliers are still -1 inside the model, so we derive their coarse topic
# from an empirical fine→coarse lookup built from non-outlier docs.

print(f'Reducing to {NR_COARSE_TOPICS} coarse topics …')
topics_coarse_raw, _ = topic_model.reduce_topics(docs, nr_topics=NR_COARSE_TOPICS)
topics_coarse_raw = np.array(topics_coarse_raw, dtype=np.int16)

fine_to_coarse = {}
for fine, coarse in zip(topics_raw, topics_coarse_raw):
    if fine != -1 and coarse != -1 and fine not in fine_to_coarse:
        fine_to_coarse[int(fine)] = int(coarse)

topics_coarse = np.array(
    [fine_to_coarse.get(int(t), t) if t != -1 else -1 for t in topics_fine],
    dtype=np.int16,
)
print(f'Coarse unmapped: {(topics_coarse == -1).sum():,}')

topic_model.save(f'{OUT_DIR}/bertopic_coarse',
                 serialization='safetensors', save_ctfidf=True)

rows = []
for _, row in topic_model.get_topic_info().iterrows():
    tid = row['Topic']
    if tid == -1:
        continue
    kws = topic_model.get_topic(tid)
    rows.append({
        'topic_id': tid, 'count': row['Count'], 'name': row.get('Name', ''),
        'keywords': ', '.join(w for w, _ in kws[:10]),
        'scores':   ', '.join(f'{s:.4f}' for _, s in kws[:10]),
    })
pd.DataFrame(rows).to_csv(f'{OUT_DIR}/topic_keywords_coarse.csv', index=False)
print(f'Saved topic_keywords_coarse.csv  ({len(rows)} topics)')

In [ ]:
# Cell 8 — Build assignments table
assignments = pd.DataFrame({
    'chunk_id':        combined['chunk_id'].values,
    'doc_id':          combined['doc_id'].values,
    'doc_type':        combined['doc_type'].values,
    'subreddit':       combined['subreddit'].values,
    'created_utc':     combined['created_utc'].values,
    'topic_id_fine':   topics_fine,
    'topic_prob_fine': probs_fine,
    'topic_id_coarse': topics_coarse,
})
assignments.to_parquet(f'{OUT_DIR}/chunk_topics.parquet', index=False)
print('Saved chunk_topics.parquet')

In [ ]:
# Cell 9 — Write topic columns back into chunk parquets
merge_cols = ['chunk_id', 'topic_id_fine', 'topic_prob_fine', 'topic_id_coarse']
for doc_type, src_path, out_name in [
    ('submission', SUBMISSIONS_CHUNKS, 'submissions_chunks.parquet'),
    ('comment',    COMMENTS_CHUNKS,    'comments_chunks.parquet'),
]:
    df = pd.read_parquet(src_path)
    subset = assignments[assignments['doc_type'] == doc_type][merge_cols]
    df = df.merge(subset, on='chunk_id', how='left', validate='1:1')
    df.to_parquet(f'{OUT_DIR}/{out_name}', index=False)
    print(f'Saved {out_name}')

print('\nStage 4 complete.')